## Conhecendo a Requests

Usando requests e fazendo requisição

In [104]:
import requests
import pandas as pd

In [105]:
r = requests.get('https://api.github.com/events')
if r.status_code == 200:
    print(r.status_code)
else:    
    print(f'Error: {r.status_code}')

200


## Extraindo dados

In [106]:
api_base_url = 'https://api.github.com'
owner = 'amzn'
url = f'{api_base_url}/users/{owner}/repos'

In [107]:
response = requests.get(url)
response.status_code

200

In [108]:
import os
from dotenv import load_dotenv
load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')
headers = {'Authorization': 'Bearer ' + access_token, 'X-GitHub-Api-Version': '2022-11-28'}

In [109]:
response = requests.get(url, headers=headers)
response.status_code


200

##  Paginando os repositórios

In [110]:
api_base_url = 'https://api.github.com'
owner = 'amzn'
url = f'{api_base_url}/users/{owner}/repos'

url

'https://api.github.com/users/amzn/repos'

In [111]:
repos_list = []

for page_num in range(1, 7):
    try:
        url_page = f'{url}?page={page_num}'
        response = requests.get(url_page, headers=headers)
        repos_list.append(response.json())
    except:
        repos_list.append(None)

## Transformando os dados

Nome dos repositórios

In [112]:
repos_name = []
for page in repos_list:
    for repo in page:
        repos_name.append(repo['name'])

In [113]:
list(repos_name[:10])

['.github',
 'ads-advanced-tools-docs',
 'ads-pao-amznjs-gtm-server-side-template',
 'ads-pao-amznjs-gtm-template',
 'alexa-coho',
 'alexa-skills-kit-js',
 'amazon-ads-advertiser-audience-normalization-sdk-py',
 'amazon-advertising-api-php-sdk',
 'amazon-codeguru-profiler-for-spark',
 'amazon-frustration-free-setup-certification-tool']

Linguagens dos repositórios

In [114]:
repos_language = []
for page in repos_list:
    for repo in page:
        repos_language.append(repo['language'])

In [115]:
list(repos_language[:10])

[None,
 'Jupyter Notebook',
 'Smarty',
 'Smarty',
 'JavaScript',
 None,
 'Python',
 'PHP',
 'Java',
 'Python']

## Criando um DataFrame

In [116]:
df = pd.DataFrame({'repository_name': repos_name, 'repository_language': repos_language})

In [117]:
df.to_csv('amazon_repos.csv', index=False)

## Armazenando repositório com POST

In [118]:
api_base_url = 'https://api.github.com'
url = f'{api_base_url}/user/repos'

url

'https://api.github.com/user/repos'

In [119]:
data = {
    'name': 'linguagens-utilizadas',
    'description': 'Repositorio com as linguagens de prog da Amazon',
    'private': False
}

In [120]:
response = requests.post(url, json=data, headers=headers)
response.status_code

422

## Formato do arquivo

In [121]:
import base64

In [122]:
with open('amazon_repos.csv', 'rb') as file:
    content = file.read()

encoded_content = base64.b64encode(content)

Upload do arquivo com PUT

In [123]:
api_base_url = 'https://api.github.com'
username = 'JoaoHHoffmann'
repo = 'linguagens-utilizadas'
path = 'amazon_repos.csv'

url = f'{api_base_url}/repos/{username}/{repo}/contents/{path}'
url

'https://api.github.com/repos/JoaoHHoffmann/linguagens-utilizadas/contents/amazon_repos.csv'

In [124]:
data = {
    'message': 'Adicionando um novo arquivo',
    'content': encoded_content.decode('utf-8')
}

In [125]:
data = {
    'message': 'Adicionando um novo arquivo',
    'content': encoded_content.decode('utf-8')
}

response = requests.put(url, json=data, headers=headers)
response.status_code

422